In [4]:
# Setting Up the Environment

import sys
import os
import time
import logging
import torch
import numpy as np
from PIL import Image
import rembg
import pymeshlab as pymesh
from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground
from tkinter.filedialog import askopenfilename
from IPython.display import Video
from pathlib import Path
from tkinter import filedialog, Tk




In [5]:
# Configure Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")


In [6]:
# Timer Utility Class
class Timer:
    def __init__(self):
        self.items = {}
        self.time_scale = 1000.0  # ms
        self.time_unit = "ms"
    
    def start(self, name: str) -> None:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        self.items[name] = time.time()
        logging.info(f"{name} starting...")

    def end(self, name: str) -> float:
        if name not in self.items:
            return
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start_time = self.items.pop(name)
        delta = time.time() - start_time
        t = delta * self.time_scale
        logging.info(f"{name} finished in {t:.2f}{self.time_unit}.")
        return t

timer = Timer()

In [7]:
# Model and Parameters Setup
device = "cuda:0" if torch.cuda.is_available() else "cpu"
pretrained_model_name_or_path = "stabilityai/TripoSR"
chunk_size = 2048  # Reduced chunk size for lower memory usage
foreground_ratio = 0.85
output_dir = "output"
model_save_format = "obj"
render = True
os.makedirs(output_dir, exist_ok=True)


In [9]:
# Initialize TripoSR Model
timer.start("Initializing model")
model = TSR.from_pretrained(
    pretrained_model_name_or_path,
    config_name="config.yaml",
    weight_name="model.ckpt",
)
model.renderer.set_chunk_size(chunk_size)
model.to(device)
timer.end("Initializing model")

2025-03-15 14:34:31,984 [INFO] Initializing model starting...
2025-03-15 14:34:40,245 [INFO] Initializing model finished in 8261.22ms.


8261.219501495361

In [6]:
# Input Images from Folder
image_folder = "images/Clay_Medium_Pots"  # Folder containing images
file_paths = list(Path(image_folder).glob("*.png")) + list(Path(image_folder).glob("*.jpg")) + list(Path(image_folder).glob("*.jpeg"))

if not file_paths:
    logging.error(f"No images found in folder: {image_folder}. Exiting.")
    sys.exit()


In [7]:
# Load and Process Multiple Images
def process_images(file_paths, foreground_ratio):
    images = []
    rembg_session = rembg.new_session()
    for file_path in file_paths:
        image = Image.open(file_path).convert("RGBA")
        image = image.resize((256, 256))  # Downscale to reduce memory usage
        image = remove_background(image, rembg_session)  # Remove background
        image = resize_foreground(image, foreground_ratio)  # Resize foreground

        # Ensure Image Mode is Correct
        if image.mode != "RGBA":
            image = image.convert("RGBA")

        image = np.array(image).astype(np.float32) / 255.0
        image = image[:, :, :3] * image[:, :, 3:4] + (1 - image[:, :, 3:4]) * 0.5
        image = Image.fromarray((image * 255.0).astype(np.uint8))
        images.append(image)
    return images


In [8]:
# Main Script for Multiple Images
timer.start("Processing images")
processed_images = process_images(file_paths, foreground_ratio)
timer.end("Processing images")


2025-01-13 10:58:06,406 [INFO] Processing images starting...
2025-01-13 10:58:21,033 [INFO] Processing images finished in 14627.70ms.


14627.701997756958

In [9]:
# Create output directory for the object
base_name = Path(image_folder).name  # Use folder name as the base name
image_dir = os.path.join(output_dir, base_name)
os.makedirs(image_dir, exist_ok=True)

In [10]:
# Save Processed Images
for idx, img in enumerate(processed_images):
    img.save(os.path.join(image_dir, f"input_processed_{idx:03d}.png"))


In [13]:
import imageio

def save_video(images, output_path, fps=30):
    """
    Save a list of PIL images as a video.

    Args:
        images (list): List of PIL images.
        output_path (str): Path to save the output video file.
        fps (int): Frames per second for the video.
    """
    with imageio.get_writer(output_path, fps=fps) as writer:
        for img in images:
            # Convert PIL image to numpy array
            frame = np.array(img)
            writer.append_data(frame)
    logging.info(f"Video saved to {output_path}")


In [14]:

# Generate 3D Model from Multiple Images in Batches
batch_size = 2  # Adjust based on available memory
scene_codes = []

logging.info("Starting the 3D model generation process.")

# Timer for running the model
timer.start("Running model")
logging.info("Running model in batches with batch size %d", batch_size)
with torch.no_grad():
    for i in range(0, len(processed_images), batch_size):
        logging.info("Processing batch %d to %d", i, min(i + batch_size, len(processed_images)))
        batch = processed_images[i:i + batch_size]
        scene_codes_batch = model(batch, device=device)
        scene_codes.append(scene_codes_batch)
timer.end("Running model")
logging.info("Model running complete.")

# Combine scene codes (Fix for the undefined 'scene_codes_combined')
scene_codes_combined = torch.cat(scene_codes, dim=0)  # Combine along batch dimension

# Rendering
if render:
    timer.start("Rendering")
    logging.info("Starting rendering phase.")
    
    # Split rendering into smaller batches
    render_images = []
    views_per_batch = 5  # Number of views per batch
    total_views = 10  # For testing, reduce total views
    logging.info("Total views: %d, Views per batch: %d", total_views, views_per_batch)
    
    for i in range(0, total_views, views_per_batch):
        logging.info("Rendering batch %d to %d", i, min(i + views_per_batch, total_views))
        batch_render_images = model.render(
            scene_codes_combined,
            n_views=min(views_per_batch, total_views - i),
            height=256,  # Lower resolution
            width=256,
            return_type="pil"
        )
        render_images.extend(batch_render_images[0])  # Collect results
    
    # Save individual images
    for ri, render_image in enumerate(render_images):
        render_path = os.path.join(image_dir, f"render_{ri:03d}.png")
        logging.info("Saving render image %d to %s", ri, render_path)
        render_image.save(render_path)
    
    # Save video
    video_path = os.path.join(image_dir, "render.mp4")
    logging.info("Saving video to %s", video_path)
    save_video(render_images, video_path, fps=30)
    timer.end("Rendering")
    logging.info("Rendering complete.")

logging.info("Processing complete.")


2025-01-13 13:01:41,843 [INFO] Starting the 3D model generation process.
2025-01-13 13:01:41,843 [INFO] Running model starting...
2025-01-13 13:01:41,843 [INFO] Running model in batches with batch size 2
2025-01-13 13:01:41,847 [INFO] Processing batch 0 to 2
2025-01-13 13:02:30,333 [INFO] Processing batch 2 to 4
2025-01-13 13:03:17,846 [INFO] Processing batch 4 to 6
2025-01-13 13:04:05,545 [INFO] Processing batch 6 to 8
2025-01-13 13:04:53,399 [INFO] Processing batch 8 to 10
2025-01-13 13:05:40,970 [INFO] Processing batch 10 to 12
2025-01-13 13:06:28,295 [INFO] Processing batch 12 to 14
2025-01-13 13:07:15,693 [INFO] Running model finished in 333850.12ms.
2025-01-13 13:07:15,693 [INFO] Model running complete.
2025-01-13 13:07:15,694 [INFO] Rendering starting...
2025-01-13 13:07:15,701 [INFO] Starting rendering phase.
2025-01-13 13:07:15,701 [INFO] Total views: 10, Views per batch: 5
2025-01-13 13:07:15,701 [INFO] Rendering batch 0 to 5
2025-01-13 14:02:46,675 [INFO] Rendering batch 5 t

In [15]:
# Combine scene codes into a single tensor
scene_codes_combined = torch.cat(scene_codes, dim=0)  # Combine along batch dimension

# Exporting Mesh
timer.start("Exporting mesh")
logging.info("Exporting mesh to file.")
try:
    meshes = model.extract_mesh(scene_codes_combined, has_vertex_color=False)  # Pass combined tensor
    mesh_file = os.path.join(image_dir, f"mesh.{model_save_format}")
    logging.info("Mesh file will be saved as: %s", mesh_file)
    meshes[0].export(mesh_file)
    logging.info("Mesh export complete.")
except Exception as e:
    logging.error(f"Error during mesh export: {e}")
finally:
    timer.end("Exporting mesh")


2025-01-13 16:43:26,638 [INFO] Exporting mesh starting...
2025-01-13 16:43:26,640 [INFO] Exporting mesh to file.
2025-01-13 17:06:50,588 [INFO] Mesh file will be saved as: output\Clay_Medium_Pots\mesh.obj
2025-01-13 17:06:52,602 [INFO] Mesh export complete.
2025-01-13 17:06:52,609 [INFO] Exporting mesh finished in 1405970.69ms.


In [ ]:
# # if to fail
# # Generate 3D Model from Multiple Images in Batches
# batch_size = 2  # Adjust based on available memory
# scene_codes = []

# timer.start("Running model")
# with torch.no_grad():
#     for i in range(0, len(processed_images), batch_size):
#         batch = processed_images[i:i + batch_size]
#         scene_codes_batch = model(batch, device=device)
#         scene_codes.append(scene_codes_batch)

#         # Save intermediate results
#         if i % 10 == 0:  # Save every 10 iterations
#             torch.save(scene_codes, f"scene_codes_checkpoint_{i}.pt")
# timer.end("Running model")

# # Rendering
# if render:
#     timer.start("Rendering")
    
#     render_images = []
#     views_per_batch = 5  # Number of views per batch
#     total_views = 10  # For testing, reduce total views

#     for i in range(0, total_views, views_per_batch):
#         batch_render_images = model.render(
#             scene_codes_combined,
#             n_views=min(views_per_batch, total_views - i),
#             height=256,  # Lower resolution
#             width=256,
#             return_type="pil"
#         )
#         render_images.extend(batch_render_images[0])  # Collect results

#         # Save intermediate results
#         if i % 2 == 0:  # Save every 2 iterations
#             for ri, render_image in enumerate(render_images):
#                 render_image.save(os.path.join(image_dir, f"render_{ri:03d}.png"))
    
#     # Save final video
#     save_video(render_images, os.path.join(image_dir, "render.mp4"), fps=30)
#     timer.end("Rendering")

# # Exporting Mesh
# timer.start("Exporting mesh")
# meshes = model.extract_mesh(scene_codes, has_vertex_color=False)

# # Save mesh
# mesh_file = os.path.join(image_dir, f"mesh.{model_save_format}")
# meshes[0].export(mesh_file)
# timer.end("Exporting mesh")

# logging.info("Processing complete.")


In [16]:
# Save as .glb
try:
    import trimesh
    mesh_trimesh = trimesh.load(mesh_file)
    glb_file = os.path.join(image_dir, "model.glb")
    mesh_trimesh.export(glb_file)
    logging.info(f"Mesh exported as .glb to {glb_file}")
except ImportError:
    logging.error("Trimesh library is required for .glb export. Install it using 'pip install trimesh'.")
except Exception as e:
    logging.error(f"Error exporting .glb: {e}")

timer.end("Exporting mesh")
logging.info("Processing complete.")


2025-01-13 17:14:31,743 [INFO] Mesh exported as .glb to output\Clay_Medium_Pots\model.glb
2025-01-13 17:14:31,748 [INFO] Processing complete.


In [17]:
# Display Render Video
Video(os.path.join(image_dir, "render.mp4"), embed=True)